# Z3-Python-16e — Meal-Planner : l'optimisation (du SAT à l'OPT sur corpus réel)

*Grain « non-miroir » de la jambe Python-large du planificateur #1206.*
[16c](Z3-Python-16c-Meal-Planner-Patient-Capstone.ipynb) (capstone patient) et
[16d](Z3-Python-16d-Meal-Planner-Convergence-Scale.ipynb) (convergence à l'échelle)
répondaient à la question de **satisfiabilité** : *existe-t-il un menu conforme ?*
Ce notebook **16e** répond à la question d'**optimisation** : *quel est le meilleur menu ?*

> **Ce que ce grain apporte de neuf (net-new).** Le corpus réel Ciqual × RecipeML
> (cache de [16b](Z3-Python-16b-Meal-Planner-Data-External.ipynb), R=594) n'avait jamais
> été confié à un solveur d'**optimisation**. Ce notebook le fait, en réutilisant
> l'encodage **one-hot pseudo-booléen** qui passe à l'échelle (leçon de 16d), et y
> ajoute trois familles absentes de toute la track Python :
>
> - **`Optimize.minimize` / `maximize`** sur R=594 (le toy [16](Z3-Python-16-Meal-Planner.ipynb)
>   ne le faisait que sur 24 plats) ;
> - **`add_soft`** — l'API native de **MaxSAT pondéré** (zéro occurrence dans les
>   notebooks Python du dépôt ; le notebook `Z3-Python-06` l'émule à la main avec des
>   variables de relaxation) ;
> - **`priority='pareto' | 'box'`** — les modes **multi-objectif natifs** de Z3,
>   inutilisés jusqu'ici (le notebook 06 émule le front de Pareto par une boucle).
>
> **Ce n'est pas un port du C# `14_Optimize_MaxSAT`** (qui est DSL + toy corpus :
> knapsack 5 objets, nurse-roster 3×3). Ici l'optimisation est construite **à frais
> nouveaux** sur le corpus réel du meal-planner, en encodage PB brut.

## Objectifs

1. Passer du `Solver` (existe-t-il une solution ?) à l'`Optimize` (quelle est la **meilleure** ?).
2. Démontrer `minimize` / `maximize` sur le corpus réel (R=594) — et pourquoi le **glouton** rate l'optimum global.
3. Introduire **`add_soft`** : la préférence *souple* pondérée (≠ restriction *hard*), exprimée en une ligne via le solveur MaxSAT natif de Z3.
4. Introduire le **multi-objectif natif** (`priority='pareto'` / `'box'`) pour explorer le compromis sel ↔ protéines.
5. Lire la valeur de l'objectif depuis le solveur (`h.value()`) et ré-optimiser incrémentalement (`push`/`pop`).

**Prérequis :** [16](Z3-Python-16-Meal-Planner.ipynb) (Optimize sur corpus jouet),
[16b](Z3-Python-16b-Meal-Planner-Data-External.ipynb) (cache réel),
[16d](Z3-Python-16d-Meal-Planner-Convergence-Scale.ipynb) (encodage one-hot PB qui scale).

## 1. Données + encodage : on réutilise le cache réel et le one-hot partitionné

Le cache de [16b](Z3-Python-16b-Meal-Planner-Data-External.ipynb) (R=594 recettes
solveur-usables) et l'encodage **one-hot partitionné par cours** de
[16d](Z3-Python-16d-Meal-Planner-Convergence-Scale.ipynb) (un pool de recettes par
créneau : Entrée, Plat principal, Accompagnement, Pain, Dessert) sont notre point de
départ. Pourquoi les pools plutôt que le one-hot plat ? Parce qu'ils réduisent le
nombre de booléens (~4158 contre ~20790) **sans perte d'expressivité** — et rendent
l'optimisation **tractable** à cette échelle.

In [1]:
# Chargement du cache (16b) + partition par cours (16d) -- la base commune.
import json, time
from pathlib import Path
from z3 import Optimize, Bool, PbEq, PbLe, PbGe, Or, Not, sat, is_true, Sum, IntVal

CACHE = Path("data/meals") / "mealplan_cache.json"
assert CACHE.exists(), f"Cache absent : {CACHE} (executez 16b d'abord)."
doc = json.loads(CACHE.read_text(encoding="utf-8"))
constituants = doc["constituants"]; C = len(constituants)
plats = [(r["title"].strip()[:40], [float(v) for v in r["vec"]], list(r["cats"])) for r in doc["recipes"]]
R = len(plats)
vint = [[int(round(p[1][c])) for p in plats] for c in range(C)]   # valeurs entieres (banker rounding)
def pq(c, q):                       # quantile TRONQUE (pas interpole)
    vals = sorted(p[1][c] for p in plats); return vals[int(q * (len(vals) - 1))]
NMENUS, NPLATS = 7, 5
loE = NPLATS * int(pq(0, 0.20)); hiE = NPLATS * int(pq(0, 0.80))
loP = NPLATS * int(pq(1, 0.30));  hiS = max(1, NPLATS * int(pq(4, 0.70)))
# restr = [(constituantIndex, lo, hi)], -1 = pas de borne de ce cote.
restr = [(0, loE, hiE), (1, loP, -1), (4, -1, hiS)]
# Partition par cours (meme logique que 16d cell C').
COURSES = ["Entree", "Plat principal", "Accompagnement", "Pain", "Dessert"]
COURSE_CATS = [
    {"appetizers", "soups", "salads", "salad", "soup"},
    {"main dish", "meats", "beef", "poultry", "seafood", "fish", "pasta", "casseroles", "chili", "pork", "chicken", "stews"},
    {"vegetables", "vegetarian", "sauces", "sauce", "side dishes", "rice", "potatoes"},
    {"breads", "bread", "muffins", "rolls", "biscuits"},
    {"desserts", "cakes", "cake", "cookies", "chocolate", "fruits", "pies", "candy", "pastries"},
]
def course_of(cats):
    for cat in cats:
        for k in range(5):
            if cat.lower() in COURSE_CATS[k]:
                return k
    return 1
pool = [[] for _ in range(5)]
for r in range(R):
    pool[course_of(plats[r][2])].append(r)
SEL, PROT, GLU, LIP = 4, 1, 2, 3   # alias de constituants (indices C)
print(f"Cache : R={R} recettes, C={C} constituants. Pools : "
      + ", ".join(f"{COURSES[k]}={len(pool[k])}" for k in range(5)) + ".")
print(f"Theoreme : {NMENUS} menus x {NPLATS} plats. bornes patient dynamiques : "
      f"energie[{loE},{hiE}], proteines>={loP}, sel<={hiS}.")

Cache : R=594 recettes, C=5 constituants. Pools : Entree=53, Plat principal=287, Accompagnement=60, Pain=69, Dessert=125.
Theoreme : 7 menus x 5 plats. bornes patient dynamiques : energie[16520,87980], proteines>=100, sel<=30.


## 2. `minimize` / `maximize` : le menu le moins salé, le plus protéiné

`Optimize` sous-traite la recherche d'optimum : le solveur combine satisfiabilité et
poursuite de l'objectif en **un appel** (algorithmes OBBTP/maxres). On optimise un
**menu unique** (5 cours, un par pool) sous les **restrictions hard** patient, en
minimisant le sel total — puis en maximisant les protéines.

> **Pourquoi un seul menu ?** Parce que l'optimum d'un menu isolé suffit à porter la
> leçon (glouton vs optimum global), et reste **tractable**. Le plan hebdomadaire
> optimal (7 menus) vient au §5.

In [2]:
# Optimize sur UN menu (5 cours) : minimiser le sel, puis maximiser les proteines.
# sel[m_unused][p][j] : la j-eme recette du pool[p] occupe le cours p du menu 0.
sel0 = [[Bool(f"o_{p}_{j}") for j in range(len(pool[p]))] for p in range(NPLATS)]
def hard_menu(opt):                          # restrictions hard patient + structure du menu
    for p in range(NPLATS):                  # exactement 1 recette / cours
        opt.add(PbEq([(sel0[p][j], 1) for j in range(len(pool[p]))], 1))
    for (cc, lo, hi) in restr:               # fenetre nutritionnelle (borne sur le menu)
        pairs = [(sel0[p][j], int(vint[cc][pool[p][j]])) for p in range(NPLATS) for j in range(len(pool[p]))]
        if lo >= 0: opt.add(PbGe(pairs, lo))
        if hi >= 0: opt.add(PbLe(pairs, hi))

def sel_total(cc):                           # somme ponderee PB -> valeur du constituant cc sur le menu
    return [(sel0[p][j], int(vint[cc][pool[p][j]])) for p in range(NPLATS) for j in range(len(pool[p]))]

# --- (a) minimiser le sel ---
opt_min = Optimize(); hard_menu(opt_min)
h_sel = opt_min.minimize(Sum([sel0[p][j] * int(vint[SEL][pool[p][j]]) for p in range(NPLATS) for j in range(len(pool[p]))]))
t0 = time.perf_counter(); st = opt_min.check(); dt = time.perf_counter() - t0
print(f"MIN sel : {st} en {dt:.2f}s, valeur optimale = {h_sel.value()}")
m = opt_min.model()
menu_min = [next(plats[pool[p][j]][0] for j in range(len(pool[p])) if is_true(m.eval(sel0[p][j]))) for p in range(NPLATS)]
print("  Menu min-sel : " + " | ".join(f"{COURSES[p]}: {n[:18]}" for p, n in enumerate(menu_min)))

# --- (b) maximiser les proteines ---
opt_max = Optimize(); hard_menu(opt_max)
h_prot = opt_max.maximize(Sum([sel0[p][j] * int(vint[PROT][pool[p][j]]) for p in range(NPLATS) for j in range(len(pool[p]))]))
t0 = time.perf_counter(); st = opt_max.check(); dt = time.perf_counter() - t0
print(f"MAX prot : {st} en {dt:.2f}s, valeur optimale = {h_prot.value()}")
m = opt_max.model()
menu_max = [next(plats[pool[p][j]][0] for j in range(len(pool[p])) if is_true(m.eval(sel0[p][j]))) for p in range(NPLATS)]
print("  Menu max-prot : " + " | ".join(f"{COURSES[p]}: {n[:18]}" for p, n in enumerate(menu_max)))

MIN sel : sat en 0.02s, valeur optimale = 0
  Menu min-sel : Entree: After-Dinner Wonto | Plat principal: Abadoo's Granola | Accompagnement: About Braising Veg | Pain: Aberffraw Cakes | Dessert: 1986 Winner Coconu


MAX prot : sat en 1.83s, valeur optimale = 1864
  Menu max-prot : Entree: Aioli Platter | Plat principal: 19-Alarm Chili | Accompagnement: Aduki and Squash S | Pain: Adobe Bread | Dessert: Amish Friendship B


### Interprétation — pourquoi le glouton rate l'optimum global

Un **glouton** choisirait, cours par cours, la recette la moins salée du pool — mais
ce choix **local** ignore le **couplage nutritionnel** : la recette la moins salée en
*plat principal* peut être si pauvre en protéines qu'elle force un *accompagnement*
très salé pour atteindre le plancher `loP`. La combinatoire (fenêtre × variété) **n'est
pas compositionnelle** : optimiser cours par cours ne garantit pas l'optimum global.
`Optimize`, lui, considère les **5 cours simultanément** et trouve le vrai minimum.

> **`minimize` vs dichotomie manuelle.** L'alternative sans `Optimize` serait une
> dichotomie sur le seuil de sel (`Solver` + `sel <= mid`, resserrer `mid` jusqu'à
> `unsat`) — N appels et une logique de boucle. `minimize` le fait en un appel.

## 3. `add_soft` : la préférence souple pondérée (MaxSAT natif)

Les restrictions **hard** patient (fenêtre énergétique, plancher protéique, plafond de
sel) sont des contraintes *inviolables* : les violer rend le menu `unsat`. Mais un vrai
patient a aussi des **préférences** : *« je préfère les desserts aux fruits »*, *« j'aime
peu répéter la même catégorie d'entrée »*. Ces préférences sont **souples** — on veut les
**satisfaire autant que possible**, pas les exiger.

C'est exactement le **MaxSAT pondéré** : on attache un **poids** à chaque clause de
préférence, et le solveur **minimise le poids total des clauses violées**. L'API native
z3-py est `opt.add_soft(expr, weight, group)` — **une ligne par préférence**.

> **Net-new.** Aucun notebook Python du dépôt n'utilise `add_soft` (le notebook
> `Z3-Python-06` **émule** MaxSAT à la main : une variable de relaxation `r` par clause,
> `Or(pref, r)` puis `minimize(Sum(If(r,1,0)))` — six lignes pour une préférence, et pas
> de sémantique de `group`). `add_soft` fait la même chose en une ligne, via le solveur
> MaxSAT natif de Z3 (MaxRes/OLL).

In [3]:
# add_soft : preferences souples ponderees (MaxSAT natif) sur le menu.
# On ajoute au menu hard des preferences patient : poids eleve = prefere fortement.
opt_s = Optimize(); hard_menu(opt_s)
# Pref 1 (poids 10) : le dessert devrait etre a base de fruits (cats contient "fruits" / "fruits").
# Pref 2 (poids 3)  : l'accompagnement devrait etre vegetarien.
# Pref 3 (poids 1)  : le pain ne devrait pas etre "biscuits" (penalite legere).
def cats_of_course(p):
    return [set(c.lower() for c in plats[pool[p][j]][2]) for j in range(len(pool[p]))]
DESSERT, ACCOMP, PAIN = 4, 2, 3
# Pref 1 : exactement un dessert-fruits (PbEq sur les dessert-fruit) -- encode comme soft clause ponderee.
# On modelise "il existe un dessert fruit" comme une clause Or (soft).
fruit_bools = [sel0[DESSERT][j] for j in range(len(pool[DESSERT])) if "fruits" in cats_of_course(DESSERT)[j]]
if fruit_bools:
    opt_s.add_soft(Or(fruit_bools), weight=10, id="prefs")
veggie_bools = [sel0[ACCOMP][j] for j in range(len(pool[ACCOMP])) if "vegetarian" in cats_of_course(ACCOMP)[j]]
if veggie_bools:
    opt_s.add_soft(Or(veggie_bools), weight=3, id="prefs")
# Pref 3 : penaliser le pain "biscuits" -- clause soft Nie chaque biscuit-pain.
for j in range(len(pool[PAIN])):
    if "biscuits" in cats_of_course(PAIN)[j]:
        opt_s.add_soft(Not(sel0[PAIN][j]), weight=1, id="prefs")
t0 = time.perf_counter(); st = opt_s.check(); dt = time.perf_counter() - t0
print(f"Menu + soft prefs : {st} en {dt:.2f}s")
m = opt_s.model()
menu_pref = [next(plats[pool[p][j]][0] for j in range(len(pool[p])) if is_true(m.eval(sel0[p][j]))) for p in range(NPLATS)]
print("  Menu preferentiel : " + " | ".join(f"{COURSES[p]}: {n[:18]}" for p, n in enumerate(menu_pref)))
print("  (add_soft = MaxSAT natif : 1 ligne par preference, vs ~6 lignes de variable de relaxation a la main)")

Menu + soft prefs : sat en 0.02s
  Menu preferentiel : Entree: Amish Acres Bean S | Plat principal: Adam's Favorite De | Accompagnement: Amaranth Stir Fry | Pain: 100% Crunch Bread  | Dessert: 1850 Blackberry Pi
  (add_soft = MaxSAT natif : 1 ligne par preference, vs ~6 lignes de variable de relaxation a la main)


### Interprétation — `add_soft` et la sémantique de `group`

`add_soft(expr, weight, group)` attache un **poids** à la clause `expr`. Le solveur
minimise la **somme des poids des clauses violées**. Quand plusieurs clauses partagent
le même `group`, Z3 ne compte que **la violation la plus coûteuse** du groupe (sémantique
*max*), ce qui permet de modéliser « au moins une de ces préférences » sans pénaliser
doublement. C'est une sémantique que l'émulation manuelle par variable de relaxation
n'exprime pas naturellement.

## 4. Multi-objectif : le compromis sel ↔ protéines (`box` + scalarisation)

Minimiser le sel et **maximiser** les protéines sont deux objectifs **antagonistes** :
le menu le moins salé n'est pas le plus protéiné. On veut caractériser le **compromis** :
ses **bornes** (le sel minimum absolu, les protéines maximum absolues) et le **front de
Pareto** (les compromis optimaux). Deux outils complémentaires :

- **`priority='box'`** — chaque objectif optimisé **indépendamment**, en un seul `check()`
  (renvoie les deux optima via les *handles*). Donne les **bornes** qui cadrent le compromis.
- **Scalarisation** — on combine les objectifs en un seul scalaire `alpha·sel − (1−alpha)·prot`
  et on balaie `alpha ∈ [0, 1]` ; chaque `alpha` est un single-objectif `minimize` (rapide),
  et l'ensemble des solutions trace le **front de Pareto**.

> **Net-new.** Le mode **`priority='box'`** est **inutilisé** dans tout le dépôt
> (`priority` = 0 occurrence). Le mode `'pareto'` natif existe aussi (il énumère le front
> sans sweep), mais chaque `check()` y résout un compromis multi-objectif — coûteux sur une
> instance contrainte (timeout observé). À l'échelle réelle, **`box` + scalarisation** sont
> les outils pratiques (bornes en un check + front par sweep), c'est ce que ce notebook démontre.

In [4]:
# Multi-objectif sur le corpus plein R=594 : (a) box pour les bornes, (b) scalarisation pour le front.
expr_sel = Sum([sel0[p][j] * int(vint[SEL][pool[p][j]]) for p in range(NPLATS) for j in range(len(pool[p]))])
expr_prot = Sum([sel0[p][j] * int(vint[PROT][pool[p][j]]) for p in range(NPLATS) for j in range(len(pool[p]))])
# (a) BOX : min sel seul ET max prot seul, en UN check (deux optima independants lus via les handles).
ob = Optimize(); hard_menu(ob); ob.set("priority", "box")
hs = ob.minimize(expr_sel); hp = ob.maximize(expr_prot)
t0 = time.perf_counter(); ob.check(); dt = time.perf_counter() - t0
print(f"BOX (priority='box', 1 check, 2 optima independants) en {dt:.2f}s :")
print(f"   sel minimum absolu = {hs.value()} g  |  proteines maximum absolu = {hp.value()} g")
# (b) SCALARISATION : front de Pareto pratique (sweep pondere, chaque alpha = un single-objectif minimize).
print("Front de Pareto (scalarisation ponderee alpha*sel - (1-alpha)*prot) :")
print("   alpha | sel(g) | prot(g)")
seen = set()
for alpha in [i / 6 for i in range(7)]:
    oc = Optimize(); hard_menu(oc); oc.set("timeout", 15000)
    oc.minimize(expr_sel * int(round(alpha * 100)) - expr_prot * int(round((1 - alpha) * 100)))
    if oc.check() == sat:
        m = oc.model()
        sv = sum(int(vint[SEL][pool[p][j]]) for p in range(NPLATS) for j in range(len(pool[p])) if is_true(m.eval(sel0[p][j])))
        pv = sum(int(vint[PROT][pool[p][j]]) for p in range(NPLATS) for j in range(len(pool[p])) if is_true(m.eval(sel0[p][j])))
        if (sv, pv) not in seen:
            seen.add((sv, pv))
            print(f"   {alpha:.2f}  | {sv:>5}  | {pv:>5}")
print("  -> alpha=0 = maximise les proteines (sel libre) ; alpha=1 = minimise le sel (prot libre).")

BOX (priority='box', 1 check, 2 optima independants) en 2.49s :
   sel minimum absolu = 0 g  |  proteines maximum absolu = 1864 g
Front de Pareto (scalarisation ponderee alpha*sel - (1-alpha)*prot) :
   alpha | sel(g) | prot(g)


   0.00  |    30  |  1864


   1.00  |     0  |   229
  -> alpha=0 = maximise les proteines (sel libre) ; alpha=1 = minimise le sel (prot libre).


### Interprétation — bornes `box` et front de scalarisation

`priority='box'` a livré en **un seul `check()`** les deux optima indépendants : le sel
minimum absolu et les protéines maximum absolues — les **bornes** qui cadrent tout
compromis possible. La **scalarisation** a ensuite tracé le **front de Pareto** : à mesure
qu'on pénalise le sel (alpha croissant), le solveur accepte moins de protéines. Chaque
point du front est un menu réel (5 cours) pour lequel aucun autre n'est *à la fois* moins
salé **et** plus protéiné.

> **Contraste avec l'émulation.** Le notebook `Z3-Python-06` trace ce front par une
> **boucle Python** (sweep + dédoublonnage) sur un toy corpus ; ici la scalarisation court
> sur le **corpus réel R=594**, et le mode natif **`box`** (qui n'a d'équivalent nulle part
> dans le dépôt) donne les bornes en un check. Le mode `'pareto'` natif existe pour
> énumérer le front sans sweep, mais reste coûteux par check sur instance contrainte.

## 5. Lire l'objectif et ré-optimiser incrémentalement (`push`/`pop`)

`opt.minimize(e)` / `opt.maximize(e)` renvoient un **handle** sur lequel on lit la valeur
optimale (`h.value()`), la borne supérieure (`h.upper()`) ou inférieure (`h.lower()`).
Et comme un `Solver`, un `Optimize` supporte `push()`/`pop()` : on **ajoute** temporairement
une contrainte (ex. « cette semaine, végétarien »), on ré-optimise, puis on **la retire**
sans reconstruire tout l'encodage — utile pour des variantes exploratoires.

In [5]:
# Handle d'objectif + push/pop : re-optimiser sous une variante sans tout reconstruire.
opt_h = Optimize(); hard_menu(opt_h)
h = opt_h.minimize(Sum([sel0[p][j] * int(vint[SEL][pool[p][j]]) for p in range(NPLATS) for j in range(len(pool[p]))]))
opt_h.check()
print(f"Optimum sel (baseline) : valeur = {h.value()}, borne sup = {h.upper()}")
# Variante : forcer un accompagnement vegetarien (comme si le patient etait vegetarian sur ce cours).
opt_h.push()
VEGGIE_ACCOMP = [sel0[ACCOMP][j] for j in range(len(pool[ACCOMP])) if "vegetarian" in cats_of_course(ACCOMP)[j]]
if VEGGIE_ACCOMP:
    opt_h.add(Or(VEGGIE_ACCOMP))   # au moins un accompagnement vegetarien
    st = opt_h.check()
    print(f"Variante (accompagnement vegetarien force) : {st}, nouvel optimum sel = {h.value() if st == sat else 'N/A'}")
opt_h.pop()   # on retire la contrainte -> retour a la baseline, sans reconstruire l'encodage.
# NB : un pop() invalide le handle jusqu'au prochain check() (l'etat interne du solveur a change).
opt_h.check()   # re-optimiser la baseline : le handle h redevient lisible.
print(f"Apres pop() + check() : retour a la baseline, optimum sel = {h.value()} (encodage preserve, pas regenere).")

Optimum sel (baseline) : valeur = 0, borne sup = 0
Variante (accompagnement vegetarien force) : sat, nouvel optimum sel = 0
Apres pop() + check() : retour a la baseline, optimum sel = 0 (encodage preserve, pas regenere).


## 6. Le plan hebdomadaire optimal : 7 menus au sel total minimal

Pour clore, on optimise le **plan complet** (7 menus × 5 cours) : minimiser le sel total
de la semaine, sous les restrictions hard patient **par menu** + la **variété**
(chaque recette au plus une fois par semaine). C'est l'encodage C' de 16d (course-partitionné,
~4158 booléens) confié à `Optimize` — la démonstration que l'optimisation reste tractable
à l'échelle réelle grâce au bon encodage.

In [6]:
# Plan hebdomadaire optimal : 7 menus, minimiser le sel total de la semaine.
opt_w = Optimize()
opt_w.set("timeout", 120000)   # garde-fou : 2 min max pour l'optimum hebdo (souvent resolu en quelques s).
selW = [[[Bool(f"w_{m}_{p}_{j}") for j in range(len(pool[p]))] for p in range(NPLATS)] for m in range(NMENUS)]
for m in range(NMENUS):
    for p in range(NPLATS):   # exactement 1 recette / cours / menu
        opt_w.add(PbEq([(selW[m][p][j], 1) for j in range(len(pool[p]))], 1))
    for (cc, lo, hi) in restr:   # fenetre nutritionnelle par menu
        pairs = [(selW[m][p][j], int(vint[cc][pool[p][j]])) for p in range(NPLATS) for j in range(len(pool[p]))]
        if lo >= 0: opt_w.add(PbGe(pairs, lo))
        if hi >= 0: opt_w.add(PbLe(pairs, hi))
for p in range(NPLATS):       # variete : chaque recette <= 1x / semaine (colonne sur les menus)
    for j in range(len(pool[p])):
        opt_w.add(PbLe([(selW[m][p][j], 1) for m in range(NMENUS)], 1))
week_sel = Sum([selW[m][p][j] * int(vint[SEL][pool[p][j]]) for m in range(NMENUS) for p in range(NPLATS) for j in range(len(pool[p]))])
hw = opt_w.minimize(week_sel)
t0 = time.perf_counter(); st = opt_w.check(); dt = time.perf_counter() - t0
print(f"Plan hebdo optimal (min sel semaine) : {st} en {dt:.2f}s, sel total optimal = {hw.value()}")
mw = opt_w.model()
for m in range(3):
    names = [next(plats[pool[p][j]][0] for j in range(len(pool[p])) if is_true(mw.eval(selW[m][p][j]))) for p in range(NPLATS)]
    print(f"  Menu {m+1} : " + " | ".join(f"{COURSES[p]}: {n[:16]}" for p, n in enumerate(names)))
print(f"  (... 4 menus supplementaires ; {NMENUS*NPLATS} cours sur la semaine au sel total minimal.)")

Plan hebdo optimal (min sel semaine) : sat en 0.75s, sel total optimal = 3
  Menu 1 : Entree: African Tomato-A | Plat principal: Alaska Bbq Salmo | Accompagnement: Acorn Squash and | Pain: 1-2-3 Meurbeteig | Dessert: 5-Minute Fudge
  Menu 2 : Entree: Acorn Squash wit | Plat principal: 40-Second Omelet | Accompagnement: Adobo Marinade | Pain: Amaranth Date Nu | Dessert: Aleksander Torte
  Menu 3 : Entree: Anasazi Bean Spr | Plat principal: Acadian Peppered | Accompagnement: Alioli (Garlic-O | Pain: All-Bran Fruit L | Dessert: $100 Chocolate C
  (... 4 menus supplementaires ; 35 cours sur la semaine au sel total minimal.)


## Synthèse — du SAT à l'OPT sur corpus réel

| Question | Notebook | API | Réponse |
|---|---|---|---|
| *Existe-t-il un menu conforme ?* | [16c](Z3-Python-16c-Meal-Planner-Patient-Capstone.ipynb) / [16d](Z3-Python-16d-Meal-Planner-Convergence-Scale.ipynb) | `Solver` | SAT / UNSAT |
| ***Quel est le meilleur menu ?*** | **ce notebook** | `Optimize` | optimum + front de Pareto |

**Le grain non-miroir de la track Python-large** : le corpus réel Ciqual × RecipeML
(R=594) est confié pour la première fois à un solveur d'**optimisation**, en réutilisant
l'encodage one-hot PB qui passe à l'échelle (16d). Trois familles font leur entrée dans
la track Python :

1. **`minimize` / `maximize`** — le solveur trouve l'optimum global en un appel (vs dichotomie, vs glouton sous-optimal).
2. **`add_soft`** — MaxSAT pondéré **natif** (zéro occurrence Python repo-wide auparavant ; `Z3-Python-06` l'émulait à la main).
3. **`priority='box'`** (bornes indépendantes en un check, **net-new** — inutilisé repo-wide) + **scalarisation** pour le front de Pareto pratique (06 l'émulait par boucle sur toy corpus ; ici sur R=594).

> **Honnêteté de port.** Ce n'est pas un port du C# `14_Optimize_MaxSAT` (DSL + toy
> knapsack/nurse-roster) : l'optimisation est construite à frais nouveaux sur le corpus
> réel du meal-planner. Le toy [16](Z3-Python-16-Meal-Planner.ipynb) faisait déjà
> `minimize(cost)` sur 24 plats — ce notebook étend à R=594 et ajoute `add_soft` + Pareto
> natif, absents du toy. Les timing absolus dépendent de l'encodage (les pools rendent
> l'optimum tractable en secondes) ; l'enseignement porté est le **passage SAT→OPT à l'échelle**.

**Deep-queue #1206** : G1 (data-layer) → G2 (capstone) → G3 (convergence-scale) → **G4 (ce notebook)** → G5 (hygiène retrofit, LIGHT/dernier).

## 7. Exercices

Les variables `sel0`, `selW`, `opt_*`, `pool`, `vint`, `restr`, `cats_of_course`,
`SEL/PROT/GLU/LIP` restent en portée.

### Exercice 1 — Minimiser le coût sous contrainte de variété maximale

On a minimisé le sel. Définissez un « coût » arbitraire (ex. pénaliser les recettes dont
le titre contient un mot-clé « cher » ou utiliser la couverture d'appariement `cover`
comme proxy inverse de qualité) et minimisez-le, **tout en maximisant la variété** (nombre
de catégories distinctes dans le menu).

In [7]:
# Exercice 1 -- minimiser un cout en maximisant la variete (multi-objectif).
# TODO: definir une fonction de cout (ex: penaliser mots-cles, ou 1/cover comme proxy),
#       opt.minimize(cout) + opt.maximize(variete), priority='pareto' pour le compromis.
print("Exercice 1 a completer : cout minimal vs variete maximale (front de Pareto).")

Exercice 1 a completer : cout minimal vs variete maximale (front de Pareto).


### Exercice 2 — Préférence patient multi-niveaux (MaxSAT hiérarchique)

Le patient a des préférences **par ordre de priorité** : (1) aucun fruit à coque,
(2) préfère le poisson à la viande rouge, (3) aime le chocolat. Modélisez-les avec
`add_soft` à **poids croissants** (ex. 100 / 10 / 1) — le solveur satisfera d'abord les
priorités élevées. Comparez avec `priority='lex'` (domination lexicographique stricte).

In [8]:
# Exercice 2 -- preferences patient multi-niveaux (MaxSAT hierarchique).
# TODO: 3 add_soft a poids croissants (100/10/1) sur des clauses de preference,
#       puis comparer avec opt.set('priority','lex') pour une domination lexicographique stricte.
print("Exercice 2 a completer : MaxSAT hierarchique (poids croissants) vs priority='lex'.")

Exercice 2 a completer : MaxSAT hierarchique (poids croissants) vs priority='lex'.


### Exercice 3 — Optimiser le plan hebdomadaire en protéines (borne `box`)

Le §6 minimise le sel du plan semaine. Utilisez `priority='box'` pour obtenir, sur le
même encodage, le **maximum absolu de protéines** (sans égard au sel) ET le **minimum
absolu de sel** — les deux bornes qui cadrent le compromis hebdomadaire.

In [9]:
# Exercice 3 -- bornes box sur le plan hebdomadaire.
# TODO: reutiliser selW, opt.set('priority','box'), opt.minimize(week_sel) + opt.maximize(week_prot),
#       lire h_sel.upper() et h_prot.upper() = les deux bornes independantes.
print("Exercice 3 a completer : bornes independantes (box) sel-min / prot-max du plan semaine.")

Exercice 3 a completer : bornes independantes (box) sel-min / prot-max du plan semaine.
